<a href="https://colab.research.google.com/github/pksX01/PySpark_Tutorials/blob/main/learnplatform_covid_impact_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Setup**

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"pksx01","key":"b4d4da30046b4f293a48f0584b6dbe6e"}'}

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json #Changing the permissions for kaggle.json. Now owner has read and write access but other users have no access

In [ ]:
!kaggle competitions download -c learnplatform-covid19-impact-on-digital-learning

1470.csv: Skipping, found more recently modified local copy (use --force to force download)
1570.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1204.csv: Skipping, found more recently modified local copy (use --force to force download)
1039.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1142.csv: Skipping, found more recently modified local copy (use --force to force download)
1444.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1536.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1179.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1052.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1705.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
1324.csv.zip: Skipping, found more recently modified

In [ ]:
import os
from zipfile import ZipFile

for filename in os.listdir('/content/'):
  if filename[-4:] == '.zip':
    print("Extracting Zip file....")
    with ZipFile(filename, 'r') as zip:
      zip.extractall()

Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....
Extracting Zip file....


In [ ]:
!pip install pyspark

     |████████████████████████████████| 281.3 MB 41 kB/s 
     |████████████████████████████████| 198 kB 21.5 MB/s 
  Created wheel for pyspark: filename=pyspark-3.2.0-py2.py3-none-any.whl size=281805912 sha256=ce175fb066b3a74c94d9b8c642d96e6c0509c911bd7a6847493c340cfa36b14c
  Stored in directory: /root/.cache/pip/wheels/0b/de/d2/9be5d59d7331c6c2a7c1b6d1a4f463ce107332b1ecd4e80718
Successfully built pyspark


In [ ]:
import pyspark
from pyspark.sql import functions as f

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[4]").appName("Covid Impact Analysis on Learn Platform").getOrCreate()

In [ ]:
spark

In [ ]:
spark.sparkContext

<SparkContext master=local[4] appName=Covid Impact Analysis on Learn Platform>

In [ ]:
districts_data = spark.read.option("header", "true").csv("/content/districts_info.csv")
products_data = spark.read.option("header", "true").csv("/content/products_info.csv")

In [ ]:
sample_engadgement_data = spark.read.option("header", "true").csv("/content/1000.csv")

In [ ]:
districts_data.show(5)

+-----------+--------+------+------------------+----------------+------------------------+--------------+
|district_id|   state|locale|pct_black/hispanic|pct_free/reduced|county_connections_ratio|  pp_total_raw|
+-----------+--------+------+------------------+----------------+------------------------+--------------+
|       8815|Illinois|Suburb|          [0, 0.2[|        [0, 0.2[|               [0.18, 1[|[14000, 16000[|
|       2685|     NaN|   NaN|               NaN|             NaN|                     NaN|           NaN|
|       4921|    Utah|Suburb|          [0, 0.2[|      [0.2, 0.4[|               [0.18, 1[|  [6000, 8000[|
|       3188|     NaN|   NaN|               NaN|             NaN|                     NaN|           NaN|
|       2238|     NaN|   NaN|               NaN|             NaN|                     NaN|           NaN|
+-----------+--------+------+------------------+----------------+------------------------+--------------+
only showing top 5 rows



In [ ]:
products_data.show(5)

+-----+--------------------+------------+---------------------+------------------+--------------------------+
|LP ID|                 URL|Product Name|Provider/Company Name|         Sector(s)|Primary Essential Function|
+-----+--------------------+------------+---------------------+------------------+--------------------------+
|13117|https://www.splas...| SplashLearn|        StudyPad Inc.|           PreK-12|      LC - Digital Lear...|
|66933|https://abcmouse.com|ABCmouse.com| Age of Learning, ...|           PreK-12|      LC - Digital Lear...|
|50479|https://www.abcya...|      ABCya!|       ABCya.com, LLC|           PreK-12|      LC - Sites, Resou...|
|92993|http://www.aleks....|       ALEKS|  McGraw-Hill PreK-12|PreK-12; Higher Ed|      LC - Digital Lear...|
|73104|https://www.achie...| Achieve3000|          Achieve3000|           PreK-12|      LC - Digital Lear...|
+-----+--------------------+------------+---------------------+------------------+--------------------------+
only showi

In [ ]:
districts_data.describe().show()

+-------+------------------+---------+------+------------------+----------------+------------------------+-------------+
|summary|       district_id|    state|locale|pct_black/hispanic|pct_free/reduced|county_connections_ratio| pp_total_raw|
+-------+------------------+---------+------+------------------+----------------+------------------------+-------------+
|  count|               233|      233|   233|               233|             205|                     219|          175|
|   mean| 5219.776824034335|      NaN|   NaN|               NaN|             NaN|                     NaN|          NaN|
| stddev|2595.7515807456075|      NaN|   NaN|               NaN|             NaN|                     NaN|          NaN|
|    min|              1000|  Arizona|  City|               NaN|             NaN|                     NaN|          NaN|
|    max|              9927|Wisconsin|  Town|          [0.8, 1[|        [0.8, 1[|                  [1, 2[|[8000, 10000[|
+-------+------------------+----

In [ ]:
products_data.describe().show()

+-------+-----------------+--------------------+------------+---------------------+--------------------+--------------------------+
|summary|            LP ID|                 URL|Product Name|Provider/Company Name|           Sector(s)|Primary Essential Function|
+-------+-----------------+--------------------+------------+---------------------+--------------------+--------------------------+
|  count|              372|                 372|         372|                  371|                 352|                       352|
|   mean|54565.79569892473|                null|        null|                 null|                null|                      null|
| stddev|26247.55143689678|                null|        null|                 null|                null|                      null|
|    min|            10533|http://appinvento...|    ABC News|  A&E Television N...|           Corporate|      CM - Classroom En...|
|    max|            99916|    https://zoom.us/| ¡Avancemos!| online-stopwat

In [ ]:
sample_engadgement_data.show(5)

+----------+-----+----------+----------------+
|      time|lp_id|pct_access|engagement_index|
+----------+-----+----------+----------------+
|2020-01-01|93690|         0|            null|
|2020-01-01|17941|      0.03|             0.9|
|2020-01-01|65358|      0.03|             1.2|
|2020-01-01|98265|      0.57|           37.79|
|2020-01-01|59257|         0|            null|
+----------+-----+----------+----------------+
only showing top 5 rows



In [ ]:
sample_engadgement_data.describe().show()

+-------+----------+------------------+-----------------+------------------+
|summary|      time|             lp_id|       pct_access|  engagement_index|
+-------+----------+------------------+-----------------+------------------+
|  count|    104003|            104001|           104003|             61655|
|   mean|      null| 54894.86647243777|0.514409103583456|192.10293682587775|
| stddev|      null|26507.416228810544|2.963189113590955|1372.6607380757682|
|    min|2020-01-01|             10067|                0|              0.26|
|    max|2020-12-31|             99984|             9.99|           9990.62|
+-------+----------+------------------+-----------------+------------------+



# **Analysis**

In [ ]:
products_data.groupBy('LP ID').count().filter(f.col('count') > 1).show()

+-----+-----+
|LP ID|count|
+-----+-----+
+-----+-----+



In [ ]:
districts_data.groupBy('district_id').count().filter(f.col('count') > 1).show()

+-----------+-----+
|district_id|count|
+-----------+-----+
+-----------+-----+



Creating a separate dataframe for each district's engadgement data, then Adding District Id to all of the engadgement data and finally combining them into a single dataframe.

In [ ]:
import os
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType

col_list = sample_engadgement_data.columns
col_list.append('district_id')
#Creating an empty dataframe for the combined engadgement data of all the districts
engadgement_data = spark.createDataFrame([], StructType([]))
#Adding a dummy row to dataframe so that it can have same schema as dataframes for each district's engadgement data will have.
for column in col_list:
    engadgement_data = engadgement_data.withColumn(column, lit('dummy'))

path = '/content/'
for filename in os.listdir(path):
  if filename[-4:] == '.csv' and filename[:-4].isdigit():
    districtId = filename.split('.')[0]
    #Creating a dataframe for each district's engadgement data
    engadgement_df = spark.read.option('header', 'true').csv(path + filename)
    #Adding District Id to the dataframe of each district's engadgement data
    engadgement_df = engadgement_df.withColumn('district_id', lit(districtId))
    #Merging all the district's engadgement data into a single dataframe
    print("Merging of Engadgement Data is in progress")
    engadgement_data = engadgement_data.union(engadgement_df)

Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress
Merging of Engadgement Data is in progress


In [ ]:
#Dropping the Dummy row from Combined Engadgement DataFrame
engadgement_data = engadgement_data.filter(f.col('district_id') != 'dummy')
#Checking the distinct count of the district id
engadgement_data.select('district_id').distinct().count()

20

In [ ]:
#Checking the no. of engadgement data files
count = 0
for filename in os.listdir(path):
  if filename[-4:] == '.csv' and filename[:-4].isdigit():
    count += 1
print(count)

20


The no. of distinct District Ids in the engadgement_data dataframe are matching with the no. of files in engadgement_data folder. It means that data from all of the files in engadgement_data folder are present in the engadgement_data dataframe.

In [ ]:
engadgement_data.count()

1851006

In [ ]:
#2nd way
'''
import os
from pyspark.sql.functions import lit

engadgement_data_list = []
path = '../input/learnplatform-covid19-impact-on-digital-learning/engagement_data/'
for filename in os.listdir(path):
    districtId = filename.split('.')[0]
    engadgement_df = spark.read.option('header', 'true').csv(path + filename)
    engadgement_df = engadgement_df.withColumn('district_id', lit(districtId))
    engadgement_data_list.append(engadgement_df)
'''

In [ ]:
#2nd way (contd)
'''num_of_df = len(engadgement_data_list)
for i in range(0, num_of_df, 2):
    curr_df = engadgement_data_list[i]
    if i < num_of_df - 2:
        next_df = engadgement_data_list[i+1]
        temp_df = curr_df.union(next_df)
        if i != 0:
            final_df = final_df.union(temp_df)
        else:
            final_df = temp_df
    else:
        final_df = final_df.union(curr_df)
'''

# **Dealing with NULL/NA values**

**Calculating the no of null values in each column of the dataframes**

In [ ]:
#Function to calculate the no of null values in each column of the dataframe
def printNullCount(df, name):
    print('No of NULL values in each column of ', name)
    for column in df.columns:
        print(column, '-->', df.filter(df[column].isNull()).count())

In [ ]:
#No of NULL values in each column of districts data
printNullCount(districts_data, 'districts_data')

No of NULL values in each column of  districts_data
district_id --> 0
state --> 0
locale --> 0
pct_black/hispanic --> 0
pct_free/reduced --> 28
county_connections_ratio --> 14
pp_total_raw --> 58


In [ ]:
#No of NULL values in each column of products data
printNullCount(products_data, 'products_data')

No of NULL values in each column of  products_data
LP ID --> 0
URL --> 0
Product Name --> 0
Provider/Company Name --> 1
Sector(s) --> 20
Primary Essential Function --> 20


In [ ]:
#No of NULL values in each column of sample engadgement data
printNullCount(sample_engadgement_data, 'sample_engadgement_data')

No of NULL values in each column of  sample_engadgement_data
time --> 0
lp_id --> 2
pct_access --> 0
engagement_index --> 42348


In [ ]:
engadgement_data.rdd.getNumPartitions()

23

In [ ]:
#printNullCount(engadgement_data, 'engadgement_data')

In [ ]:
districts_data.count()

233

In [ ]:
products_data.count()

372

In [ ]:
sample_engadgement_data.count()

104003

**Dropping the rows having NULL values in each DataFrame**

In [ ]:
#Dropping the rows having NULL values
districts_data_no_null = districts_data.dropna()
products_data_no_null = products_data.dropna()

In [ ]:
engadgement_data_no_null = engadgement_data.dropna()

In [ ]:
districts_data_no_null.describe().show()

+-------+------------------+---------+------+------------------+----------------+------------------------+-------------+
|summary|       district_id|    state|locale|pct_black/hispanic|pct_free/reduced|county_connections_ratio| pp_total_raw|
+-------+------------------+---------+------+------------------+----------------+------------------------+-------------+
|  count|               145|      145|   145|               145|             145|                     145|          145|
|   mean| 5237.806896551724|      NaN|   NaN|               NaN|             NaN|                     NaN|          NaN|
| stddev|2588.8799843876923|      NaN|   NaN|               NaN|             NaN|                     NaN|          NaN|
|    min|              1039|  Florida|  City|               NaN|             NaN|                     NaN|          NaN|
|    max|              9899|Wisconsin|  Town|          [0.8, 1[|        [0.8, 1[|               [0.18, 1[|[8000, 10000[|
+-------+------------------+----

In [ ]:
products_data_no_null.describe().show()

+-------+------------------+--------------------+------------+---------------------+--------------------+--------------------------+
|summary|             LP ID|                 URL|Product Name|Provider/Company Name|           Sector(s)|Primary Essential Function|
+-------+------------------+--------------------+------------+---------------------+--------------------+--------------------------+
|  count|               352|                 352|         352|                  352|                 352|                       352|
|   mean| 54250.03977272727|                null|        null|                 null|                null|                      null|
| stddev|26431.322695620234|                null|        null|                 null|                null|                      null|
|    min|             10533|http://appinvento...|    ABC News|  A&E Television N...|           Corporate|      CM - Classroom En...|
|    max|             99916|    https://zoom.us/| ¡Avancemos!| online

In [ ]:
engadgement_data_no_null.describe().show()

+-------+----------+-----------------+------------------+-----------------+------------------+
|summary|      time|            lp_id|        pct_access| engagement_index|       district_id|
+-------+----------+-----------------+------------------+-----------------+------------------+
|  count|   1335927|          1335927|           1335927|          1335927|           1335927|
|   mean|      null|55065.75811028597|0.8317743484485485|199.2145630936253|1361.9960229862859|
| stddev|      null| 26700.9938044692| 4.062909156988108|1651.030731006726|247.54472451218817|
|    min|2020-01-01|            10006|                 0|             0.05|              1000|
|    max|2020-12-31|            99984|              9.99|          9999.04|              1705|
+-------+----------+-----------------+------------------+-----------------+------------------+



In [ ]:
engadgement_data_no_null.count()

1335927

# **Dealing with Duplicates**

**Finding out the duplicate rows in Dataframe**

In [ ]:
from pyspark.sql import functions as f
# 1st way
districts_data_no_null.groupBy(districts_data_no_null.columns).agg(f.count('*').alias('cnt')).filter(f.col('cnt')>1).show()

+-----------+-----+------+------------------+----------------+------------------------+------------+---+
|district_id|state|locale|pct_black/hispanic|pct_free/reduced|county_connections_ratio|pp_total_raw|cnt|
+-----------+-----+------+------------------+----------------+------------------------+------------+---+
+-----------+-----+------+------------------+----------------+------------------------+------------+---+



In [ ]:
# 2nd way
products_data_no_null.groupBy(products_data_no_null.columns).count().filter('count > 1').show()

+-----+---+------------+---------------------+---------+--------------------------+-----+
|LP ID|URL|Product Name|Provider/Company Name|Sector(s)|Primary Essential Function|count|
+-----+---+------------+---------------------+---------+--------------------------+-----+
+-----+---+------------+---------------------+---------+--------------------------+-----+



In [ ]:
# 3rd way
engadgement_data_no_null.groupBy(engadgement_data_no_null.columns).count().filter(f.col('count') > 1).show()

+----+-----+----------+----------------+-----------+-----+
|time|lp_id|pct_access|engagement_index|district_id|count|
+----+-----+----------+----------------+-----------+-----+
+----+-----+----------+----------------+-----------+-----+

